# Kronos LoRA Fine-Tuning – Google Colab

Fine-tunes **NeoQuasar/Kronos-base** on energy-sector financial data (2010–2018) using LoRA.
LoRA hyperparameters are set to the values identified as optimal in the sensitivity analysis.

| Parameter | Value | Source |
|---|---|---|
| `lora_r` | 8 | Sensitivity analysis |
| `lora_alpha` | 16 | Sensitivity analysis |
| `lora_dropout` | 0.05 | Sensitivity analysis |
| `learning_rate` | 1e-4 | Sensitivity analysis |
| `max_steps` | 1000 | Training config |

**Workflow:**
1. Setup environment
2. Fine-tune Kronos
3. Evaluate on test data (2021+)
4. Download adapter weights + results

## Setup: Clone Repository

In [10]:
!rm -rf /content/BA
print("Cleaned up.")

Cleaned up.


In [11]:
import os
from pathlib import Path

repo_url = "https://github.com/bp571/BA"

%cd /content
!git clone {repo_url}
%cd /content/BA

TIINGO_API_KEY = "312c6dab6a1fe6258bbc6652bcdec49a14ee08ad"
os.environ["TIINGO_API_KEY"] = TIINGO_API_KEY
Path(".env").write_text(f"TIINGO_API_KEY={TIINGO_API_KEY}\n")
print("✅ API key configured")

!git clone https://github.com/shiyu-coder/Kronos.git 02_finetuning/models/Kronos
print("✅ Kronos cloned")

/content
Cloning into 'BA'...
remote: Enumerating objects: 537, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 537 (delta 0), reused 2 (delta 0), pack-reused 529 (from 1)
Receiving objects: 100% (537/537), 13.39 MiB | 31.68 MiB/s, done.
Resolving deltas: 100% (236/236), done.
/content/BA
✅ API key configured
Cloning into '02_finetuning/models/Kronos'...
remote: Enumerating objects: 371, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 371 (delta 6), reused 3 (delta 3), pack-reused 361 (from 2)
Receiving objects: 100% (371/371), 9.32 MiB | 23.86 MiB/s, done.
Resolving deltas: 100% (174/174), done.
✅ Kronos cloned


## Install Dependencies

In [12]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn tqdm einops \
    peft transformers huggingface_hub \
    gluonts python-dotenv tiingo

In [13]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


## Verify Training Data

In [14]:
from pathlib import Path

required = ["data/processed/train_data_kronos.arrow"]

all_ok = True
for p in required:
    path = Path(p)
    if path.exists():
        size_mb = path.stat().st_size / 1024 / 1024
        print(f"  ✅ {p}  ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ {p}  NOT FOUND")
        all_ok = False

if not all_ok:
    raise FileNotFoundError("Missing arrow file. Push it to the repo first.")

  ✅ data/processed/train_data_kronos.arrow  (1.6 MB)


## Fine-Tuning

Trains for 1000 steps with gradient accumulation (effective batch size = 8).
Checkpoints are saved every 250 steps to `models/kronos-lora-finetuned/`.

**Estimated time on A100: ~15–20 min**

In [15]:
!python 02_finetuning/training/train_kronos_lora.py

Device: cuda

Loading tokenizer: NeoQuasar/Kronos-Tokenizer-base
config.json: 100% 301/301 [00:00<00:00, 1.88MB/s]
model.safetensors: 100% 15.8M/15.8M [00:00<00:00, 19.3MB/s]
Loading model: NeoQuasar/Kronos-base
config.json: 100% 228/228 [00:00<00:00, 1.59MB/s]
model.safetensors: 100% 409M/409M [00:02<00:00, 170MB/s] 

Applying LoRA:
  Rank: 8
  Alpha: 16
  Target modules: 48
trainable params: 638,976 || all params: 102,949,568 || trainable%: 0.6207

Training:
  Max steps: 1000
  Batch size: 4
  Gradient accumulation: 2
  Output: /content/BA/models/kronos-lora-finetuned

Step 10/1000: Loss=3.6700, S1=5.7114, S2=8.9687
Step 20/1000: Loss=3.7382, S1=5.8369, S2=9.1157
Step 30/1000: Loss=3.5298, S1=4.9447, S2=9.1746
Step 40/1000: Loss=3.6665, S1=5.5411, S2=9.1247
Step 50/1000: Loss=3.5870, S1=5.5716, S2=8.7764
Step 60/1000: Loss=3.7432, S1=5.8028, S2=9.1700
Step 70/1000: Loss=3.4877, S1=4.9228, S2=9.0279
Step 80/1000: Loss=3.5654, S1=5.4635, S2=8.7981
Step 90/1000: Loss=3.4697, S1=5.1506, 

In [16]:
# Verify adapter was saved
adapter_path = Path("models/kronos-lora-finetuned/final")
if adapter_path.exists():
    files = list(adapter_path.iterdir())
    print(f"✅ Adapter saved: {adapter_path}")
    for f in files:
        print(f"   {f.name}  ({f.stat().st_size / 1024:.0f} KB)")
else:
    print("❌ Adapter not found — check training output above")

✅ Adapter saved: models/kronos-lora-finetuned/final
   README.md  (5 KB)
   adapter_model.safetensors  (2508 KB)
   adapter_config.json  (1 KB)


## Evaluation on Test Data (2021+)

Runs the rolling benchmark on all energy-sector assets using the fine-tuned adapter.
Uses the optimal inference parameters from the data parameter sensitivity analysis:
`context=120, forecast=6`.

In [17]:
!python 02_finetuning/evaluation/main_kronos_finetuned.py \
    --adapter-path models/kronos-lora-finetuned/final \
    --seed 42

✅ Random Seeds auf 42 gesetzt für Reproduzierbarkeit
Loading fine-tuned Kronos model from: models/kronos-lora-finetuned/final
config.json: 100% 301/301 [00:00<00:00, 1.95MB/s]
model.safetensors: 100% 15.8M/15.8M [00:00<00:00, 19.3MB/s]
config.json: 100% 228/228 [00:00<00:00, 1.51MB/s]
model.safetensors: 100% 409M/409M [00:03<00:00, 136MB/s] 
Loading Kronos LoRA adapter: models/kronos-lora-finetuned/final
Loading assets: 100% 34/34 [00:00<00:00, 40.63it/s]
📊 Total windows to process: 4080 across 34 assets
Processing windows:   0% 0/4080 [00:00<?, ?it/s]Processing batch group 1/1: context=120, forecast=6, n_windows=48
Processing windows:   1% 48/4080 [00:01<01:39, 40.56it/s]Processing batch group 1/1: context=120, forecast=6, n_windows=48
Processing windows:   2% 96/4080 [00:01<01:19, 49.84it/s]Processing batch group 1/1: context=120, forecast=6, n_windows=48
Processing windows:   4% 144/4080 [00:02<01:13, 53.74it/s]Processing batch group 1/1: context=120, forecast=6, n_windows=48
Proces

## Results Summary

In [18]:
import json
import pandas as pd
from pathlib import Path

results_path = Path("02_finetuning/results/kronos_finetuned/seed_42/final_energy_study.json")

with open(results_path) as f:
    data = json.load(f)

summary = data['summary']
rows = []
for ticker, metrics in summary.items():
    rows.append({
        'Ticker': ticker,
        'MAE': round(metrics.get('MAE_indicative', float('nan')), 4),
        'RankIC': round(metrics.get('RankIC_TimeSeries_Mean', float('nan')), 4),
        'IC': round(metrics.get('IC_Mean', float('nan')), 4),
    })

df = pd.DataFrame(rows).sort_values('RankIC', ascending=False)

print(f"Assets evaluated: {len(df)}")
print(f"Mean MAE:    {df.MAE.mean():.4f}")
print(f"Mean RankIC: {df.RankIC.mean():.4f}")
print(f"Mean IC:     {df.IC.mean():.4f}")
print()
display(df)

Assets evaluated: 34
Mean MAE:    2.9814
Mean RankIC: 0.0290
Mean IC:     nan



,Ticker,MAE,RankIC,IC
28,DUK,1.9849,0.1129,NaN
22,SEDG,15.5662,0.0757,NaN
23,RUN,2.2628,0.0724,NaN
7,XOM,2.5780,0.0719,NaN
2,TAN,3.0648,0.0690,NaN
12,BP,0.8618,0.0676,NaN
11,MPC,3.7945,0.0614,NaN
9,COP,3.4532,0.0595,NaN
18,HES,4.6423,0.0567,NaN
10,SLB,1.7668,0.0548,NaN


## Download Adapter + Results

In [19]:
from google.colab import files
import zipfile
from pathlib import Path

zip_name = "kronos_lora_finetuned.zip"

with zipfile.ZipFile(zip_name, 'w') as zf:
    # LoRA adapter weights
    adapter_dir = Path("models/kronos-lora-finetuned/final")
    for f in adapter_dir.iterdir():
        zf.write(f, arcname=f"adapter/{f.name}")

    # Evaluation results
    results_dir = Path("02_finetuning/results/kronos_finetuned/seed_42")
    for f in results_dir.iterdir():
        if f.suffix == '.json':
            zf.write(f, arcname=f"results/{f.name}")

files.download(zip_name)
print(f"Downloaded: {zip_name}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: kronos_lora_finetuned.zip
